# Q1: Whisper Fine-Tuning & Error Analysis

Run this in **Google Colab** with GPU runtime (A100/V100 recommended).

**Steps:**
1. Install dependencies
2. Download & preprocess dataset
3. Fine-tune Whisper-Small on Hindi data
4. Evaluate on FLEURS Hindi test set
5. Error analysis & taxonomy
6. Implement one fix with before/after results

## Install Dependencies

In [15]:
# Force-reload updated source modules
import importlib
import src.data_utils
import src.whisper_trainer
importlib.reload(src.data_utils)
importlib.reload(src.whisper_trainer)

from src.data_utils import (
    download_dataset, parse_transcription_json, segment_audio,
    build_hf_dataset, normalize_hindi_text, dataset_stats
)
from src.whisper_trainer import (
    get_processor, prepare_dataset, train_whisper,
    evaluate_on_fleurs, transcribe_audio
)
print("✓ Modules reloaded with latest changes")


✓ Modules reloaded with latest changes


In [3]:
!pip install -q torch torchaudio transformers datasets accelerate evaluate
!pip install -q librosa soundfile jiwer tqdm indic-nlp-library
!pip install -q git+https://github.com/openai/whisper.git

In [4]:
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import unicodedata
import re

In [5]:
# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.data_utils import (
    download_dataset, parse_transcription_json, segment_audio,
    build_hf_dataset, normalize_hindi_text, dataset_stats
)
from src.whisper_trainer import (
    get_processor, prepare_dataset, train_whisper,
    evaluate_on_fleurs, transcribe_audio
)

print(f"Project root: {PROJECT_ROOT}")

INFO:datasets:PyTorch version 2.10.0 available.


Project root: c:\Users\rajee\OneDrive\Desktop\JoshTech tasks


# Download & Explore Dataset

In [6]:
CSV_PATH = os.path.join(PROJECT_ROOT, "FT Data - data.csv")
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")

# Print dataset stats
stats = dataset_stats(CSV_PATH)
print(f"\nTotal recordings: {stats['num_recordings']}")
print(f"Total duration: {stats['total_duration_hours']} hours")
print(f"Unique speakers: {stats['num_unique_users']}")

INFO:src.data_utils:Dataset Stats:
INFO:src.data_utils:  num_recordings: 104
INFO:src.data_utils:  num_unique_users: 102
INFO:src.data_utils:  total_duration_seconds: 78790
INFO:src.data_utils:  total_duration_hours: 21.89
INFO:src.data_utils:  avg_duration_seconds: 757.6
INFO:src.data_utils:  min_duration_seconds: 438
INFO:src.data_utils:  max_duration_seconds: 1194



Total recordings: 104
Total duration: 21.89 hours
Unique speakers: 102


In [7]:
# Download all files (transcriptions first for analysis, then audio)
print("Downloading transcriptions and metadata...")
df = download_dataset(
    CSV_PATH, DATA_DIR,
    download_audio=True,
    download_transcription=True,
    download_metadata=True
)

print(f"\n✓ Downloaded {len(df)} recordings")


✓ Downloaded 104 recordings


## Segment Audio into Utterances

In [8]:
SEGMENTS_DIR = os.path.join(PROJECT_ROOT, "data", "processed", "segments")
all_segments = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Segmenting"):
    trans_path = row['local_transcription_path']
    audio_path = row['local_audio_path']
    
    if not os.path.exists(trans_path) or not os.path.exists(audio_path):
        continue
    
    # Parse transcription JSON
    segments_data = parse_transcription_json(trans_path)
    
    if not segments_data:
        print(f"  Warning: No segments found in {trans_path}")
        continue
    
    # Segment audio
    seg_output_dir = os.path.join(SEGMENTS_DIR, str(row['recording_id']))
    segments = segment_audio(
        audio_path, segments_data, seg_output_dir,
        min_duration=1.0, max_duration=30.0
    )
    all_segments.extend(segments)

print(f"\n✓ Created {len(all_segments)} valid segments")
print(f"  Duration range: 1-30 seconds each")

# Sample check
if all_segments:
    sample = all_segments[0]
    print(f"\n  Sample segment:")
    print(f"    Text: {sample['text'][:80]}...")
    print(f"    Duration: {sample['duration']:.1f}s")
    print(f"    Path: {sample['audio_path']}")

Segmenting: 100%|██████████| 104/104 [01:41<00:00,  1.02it/s]


✓ Created 4929 valid segments
  Duration range: 1-30 seconds each

  Sample segment:
    Text: अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको दे...
    Duration: 14.3s
    Path: c:\Users\rajee\OneDrive\Desktop\JoshTech tasks\data\processed\segments\825780\825780_seg0000.wav


## Text Normalization

In [9]:
print(f"Normalizing text for {len(all_segments)} segments...")
for seg in all_segments:
    seg["text"] = normalize_hindi_text(seg["text"])

# Filter out segments that became empty after normalization
before_count = len(all_segments)
all_segments = [seg for seg in all_segments if seg["text"].strip()]
after_count = len(all_segments)
print(f"  Kept {after_count}/{before_count} segments (removed {before_count - after_count} empty)")


Normalizing text for 4929 segments...
  Kept 4929/4929 segments (removed 0 empty)


## Build HuggingFace Dataset

In [10]:
!pip install -q "datasets==2.21.0" soundfile librosa


In [11]:
# Train & Test split in the ratio of 9:1
train_dataset, val_dataset = build_hf_dataset(all_segments, train_ratio=0.9)
print(f"  Train: {len(train_dataset)} segments")
print(f"  Val:   {len(val_dataset)} segments")

# Preprocess for Whisper
MODEL_NAME = "openai/whisper-small"
processor, feature_extractor, tokenizer = get_processor(MODEL_NAME)

print("\nPreprocessing datasets for Whisper...")
train_dataset = train_dataset.map(
    lambda batch: prepare_dataset(batch, processor),
    remove_columns=train_dataset.column_names,
)
val_dataset = val_dataset.map(
    lambda batch: prepare_dataset(batch, processor),
    remove_columns=val_dataset.column_names,
)
print("✓ Preprocessing complete")


INFO:src.data_utils:Train: 4364 segments from 93 recordings
INFO:src.data_utils:Val: 565 segments from 11 recordings


  Train: 4364 segments
  Val:   565 segments


INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://hug


Preprocessing datasets for Whisper...


Map:   0%|          | 0/4364 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

✓ Preprocessing complete


## Fine-Tune Whisper-Small

In [ ]:
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models", "whisper-small-hi")

trainer, model, processor = train_whisper(
    train_dataset,
    val_dataset,
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR,
    num_train_epochs=7,
    per_device_train_batch_size=16,
    learning_rate=1e-5,
    warmup_steps=500,
    eval_steps=500,
)

print(f"\n✓ Training complete! Model saved to {OUTPUT_DIR}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://hug

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/generation_config.json "HTTP/1.1 200 OK"
INFO:src.whisper_trainer:✅ Gradient checkpointing enabled (saves ~50% VRAM)
INFO:src.whisper_trainer:📋 Training config: epochs=7, effective_batch=32, fp16=False, bf16=False, tf32=False, compile=False, workers=0
INFO:src.whisper_trainer:✅ Early stopping enabled (patience=3 evals)
INFO:src.whisper_trainer:✅ Encoder will be frozen for first 1 epoch(s)
INFO:src.whisper_trainer:🚀 Starting optimized training...
INFO:src.whisper_trainer:🧊 Encoder FROZEN. Trainable: 153,580,800 / 241,734,912 params (63.5%). Will unfreeze after epoch 1.
c:\Users\rajee\OneDrive\Desktop\JoshTech tasks\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument 

Step,Training Loss,Validation Loss


INFO:src.whisper_trainer:🔥 Encoder UNFROZEN at epoch 1. Trainable: 241,734,912 / 241,734,912 params (100.0%).


## STEP 5: Evaluate on FLEURS Hindi Test Set

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# A) Pretrained baseline
print("--- Pretrained Whisper-Small (Baseline) ---")
pretrained_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
pretrained_processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="Hindi", task="transcribe"
)

pretrained_results = evaluate_on_fleurs(pretrained_model, pretrained_processor)
print(f"Pretrained WER: {pretrained_results['overall_wer']:.4f}")

In [ ]:
# B) Fine-tuned model
print("--- Fine-Tuned Whisper-Small ---")
finetuned_model = WhisperForConditionalGeneration.from_pretrained(OUTPUT_DIR)
finetuned_processor = WhisperProcessor.from_pretrained(OUTPUT_DIR)

finetuned_results = evaluate_on_fleurs(finetuned_model, finetuned_processor)
print(f"Fine-tuned WER: {finetuned_results['overall_wer']:.4f}")

In [ ]:
# WER Results Table
print("=" * 60)
print("WER RESULTS TABLE")
print("=" * 60)
print(f"{'Model':<35} {'Hindi WER':>10}")
print("-" * 50)
print(f"{'Whisper Small (Pretrained)':<35} {pretrained_results['overall_wer']:>10.4f}")
print(f"{'FT Whisper Small (yours)':<35} {finetuned_results['overall_wer']:>10.4f}")
print(f"{'Improvement':<35} {pretrained_results['overall_wer'] - finetuned_results['overall_wer']:>10.4f}")

# Save results
results_csv = os.path.join(PROJECT_ROOT, "results", "wer_results.csv")
os.makedirs(os.path.dirname(results_csv), exist_ok=True)
pd.DataFrame([
    {'Model': 'Whisper Small (Pretrained)', 'Hindi_WER': pretrained_results['overall_wer']},
    {'Model': 'FT Whisper Small (yours)', 'Hindi_WER': finetuned_results['overall_wer']},
]).to_csv(results_csv, index=False)
print(f"\n✓ Results saved to {results_csv}")

## STEP 6: Error Analysis (Stratified Sampling)

In [ ]:
print("=" * 60)
print("STEP 6: Error Analysis (Stratified Sampling)")
print("=" * 60)

# Use fine-tuned model results
sentence_wers = finetuned_results['sentence_wers']
predictions = finetuned_results['predictions']
references = finetuned_results['references']

# Create error analysis DataFrame
error_df = pd.DataFrame({
    'reference': references,
    'prediction': predictions,
    'wer': sentence_wers,
})

# Stratified sampling
strata = {
    'Very High (WER > 0.8)': error_df[error_df['wer'] > 0.8],
    'High (0.5 < WER ≤ 0.8)': error_df[(error_df['wer'] > 0.5) & (error_df['wer'] <= 0.8)],
    'Medium (0.3 < WER ≤ 0.5)': error_df[(error_df['wer'] > 0.3) & (error_df['wer'] <= 0.5)],
    'Low (0.1 < WER ≤ 0.3)': error_df[(error_df['wer'] > 0.1) & (error_df['wer'] <= 0.3)],
    'Very Low (WER ≤ 0.1)': error_df[(error_df['wer'] > 0) & (error_df['wer'] <= 0.1)],
}

sampled = []
for stratum_name, stratum_df in strata.items():
    if len(stratum_df) == 0:
        continue
    n_samples = min(5, len(stratum_df))
    sample = stratum_df.sample(n=n_samples, random_state=42)
    sample['stratum'] = stratum_name
    sampled.append(sample)
    print(f"  {stratum_name}: {len(stratum_df)} total → sampled {n_samples}")

sampled_df = pd.concat(sampled, ignore_index=True)
print(f"\n  Total sampled errors: {len(sampled_df)}")

# Save error analysis
error_analysis_path = os.path.join(PROJECT_ROOT, "results", "error_analysis.csv")
sampled_df.to_csv(error_analysis_path, index=False, encoding='utf-8-sig')
print(f"  Saved to: {error_analysis_path}")

In [ ]:
# Print sample errors
print("--- SAMPLE ERRORS ---")
for idx, row in sampled_df.head(10).iterrows():
    print(f"\n  [{row['stratum']}] WER: {row['wer']:.3f}")
    print(f"  REF: {row['reference'][:80]}")
    print(f"  HYP: {row['prediction'][:80]}")

## STEP 7: Error Taxonomy

In [ ]:
print("=" * 60)
print("STEP 7: Build Error Taxonomy")
print("=" * 60)

print("""
┌─────────────────────────────────────────────────────────────┐
│ ERROR TAXONOMY (to be refined after examining samples)      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ 1. CODE-SWITCHING ERRORS                                    │
│    English words in Hindi context misrecognized              │
│                                                             │
│ 2. HOMOPHONE CONFUSION                                      │
│    Similar-sounding Hindi words swapped                      │
│                                                             │
│ 3. WORD BOUNDARY ERRORS                                     │
│    Words incorrectly merged or split                         │
│                                                             │
│ 4. RARE/PROPER NOUN ERRORS                                  │
│    Uncommon names or terms misrecognized                     │
│                                                             │
│ 5. DISFLUENCY HANDLING                                      │
│    Fillers (अ, उम्म, हम्म) misinterpreted                   │
│                                                             │
│ 6. NUMBER/QUANTITY ERRORS                                    │
│    Numerical expressions mishandled                          │
│                                                             │
│ → Manually classify each sampled error into these categories │
│ → Add 3-5 examples per category with reasoning              │
│ → Save to results/error_taxonomy.md                         │
└─────────────────────────────────────────────────────────────┘
""")

print("\n✓ Q1 Complete! Review results/ directory for outputs.")
print("Next: Run Q2 notebook for ASR cleanup pipeline.")